# E05 — Full-Context Qwen Baseline (A1) — Analysis

**Research question**: How well does `qwen2.5:7b-instruct` classify NDA requirements when given
the full NDA text, with retrieval removed from the pipeline?

**A1 architecture**: full NDA document text + hypothesis -> `classification_prompt_v1` (=P0,
system prompt byte-for-byte unchanged) -> `qwen2.5:7b-instruct-ctx16k` -> label + explicit
evidence. No retrieval, no reranker, no agent, no tools. The user-message context header is
"Full NDA text:" instead of the frozen prompt's own "Retrieved NDA excerpts:" — a documented
architecture wrapper, not a new prompt variant; `classification_prompt_v1.yaml` itself was never
modified.

Manifest: `TRAIN_ARCH_v1` (150 cases, 50/50/50, seed=700, 78 unique documents, zero overlap with
`TRAIN_PROMPT_v1`) — shared with the future E07 for a matched comparison.

Local-only, $0, zero hosted calls.

In [1]:
import csv
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "E05_full_context" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
E05 = REPO_ROOT / "experiments/E05_full_context"
RESULTS = E05 / "results"

manifest = json.load(open(E05 / "TRAIN_ARCH_v1.json"))
summary = json.load(open(RESULTS / "run_E05_A1_train.json"))
wall = json.load(open(RESULTS / "run_E05_A1_train_wall_seconds.json"))
print("n_cases:", summary["n_cases"])
print("total wall time (min):", round(wall["total_wall_seconds"]/60, 1))

n_cases: 150
total wall time (min): 31.9


## 3. Context-window fix / Modelfile verification

In [2]:
print("Model tag: qwen2.5:7b-instruct-ctx16k")
print("Modelfile: configs/ollama/qwen2.5-7b-instruct-ctx16k.Modelfile (PARAMETER num_ctx 16384)")
print("Verified during calibration via `ollama ps` (CONTEXT=16384) and via input-token-usage")
print("checks matching full document length -- NOT via ModelGateway's num_ctx kwarg, which was")
print("empirically found ineffective for Ollama's OpenAI-compatible endpoint.")

Model tag: qwen2.5:7b-instruct-ctx16k
Modelfile: configs/ollama/qwen2.5-7b-instruct-ctx16k.Modelfile (PARAMETER num_ctx 16384)
Verified during calibration via `ollama ps` (CONTEXT=16384) and via input-token-usage
checks matching full document length -- NOT via ModelGateway's num_ctx kwarg, which was
empirically found ineffective for Ollama's OpenAI-compatible endpoint.


## 4. TRAIN_ARCH_v1 manifest -- verified

In [3]:
from collections import Counter
balance = Counter(c["gold_label"] for c in manifest["cases"])
print("class balance:", dict(balance))
assert balance == Counter({"Entailment": 50, "Contradiction": 50, "NotMentioned": 50})
assert manifest["seed"] == 700
assert manifest["total_unique_documents"] == 78
assert manifest["verified_zero_overlap_with_TRAIN_PROMPT_v1"] is True
print("Verified: 150 cases, 50/50/50, seed=700, 78 unique documents, zero overlap with TRAIN_PROMPT_v1")

class balance: {'Contradiction': 50, 'Entailment': 50, 'NotMentioned': 50}
Verified: 150 cases, 50/50/50, seed=700, 78 unique documents, zero overlap with TRAIN_PROMPT_v1


## 5. Full-input token distribution

In [4]:
tok = summary["input_tokens"]
print("mean:", round(tok["mean"], 1) if tok["mean"] else None)
print("median:", tok["median"])
print("p90:", tok["p90"])
print("max:", tok["max"])

mean: 2252.7
median: 2045.5
p90: 3945
max: 5806


## 6-7. Classification metrics and confusion matrix

In [5]:
cls = summary["classification"]
print(f"Accuracy:  {cls['accuracy']:.4f}")
print(f"Macro-F1:  {cls['macro_f1']:.4f}")
print("Per-class recall:")
for label, r in cls["per_class_recall"].items():
    print(f"  {label:>13}: {r:.4f}")
print()
cm = cls["confusion_matrix"]
print(f"Confusion matrix (rows=gold, cols=predicted, order={cm['labels']}):")
for label, row in zip(cm["labels"], cm["matrix"]):
    print(f"  {label:>13}: {row}")

Accuracy:  0.4000
Macro-F1:  0.3974
Per-class recall:
     Entailment: 0.4800
  Contradiction: 0.2800
   NotMentioned: 0.4400

Confusion matrix (rows=gold, cols=predicted, order=['Entailment', 'Contradiction', 'NotMentioned']):
     Entailment: [24, 17, 9]
  Contradiction: [5, 14, 31]
   NotMentioned: [20, 8, 22]


## 8. Contradiction Recall — headline metric, shown separately

In [6]:
cls = summary["classification"]
lo, hi = cls["contradiction_recall_ci95"]
print(f"Contradiction Recall: {cls['contradiction_recall']:.4f} "
      f"(n={cls['contradiction_n']}, 95% CI [{lo:.4f}, {hi:.4f}])")

Contradiction Recall: 0.2800 (n=50, 95% CI [0.1747, 0.4167])


## 9. Strict vs. recovered parse validity — never collapsed into one number

In [7]:
so = summary["structured_output"]
print(f"Strict:    {so['strict_n']}/{summary['n_cases']}  ({so['strict_parse_validity_pct']:.1f}%)")
print(f"Recovered: {so['recovered_n']}/{summary['n_cases']}")
print(f"Invalid:   {so['invalid_n']}/{summary['n_cases']}")
print(f"Usable structured-output validity (strict+recovered): {so['usable_parse_validity_pct']:.1f}%")
print(f"Total retries: {so['total_retries']}   Model errors: {so['model_errors']}   Timeouts: {so['timeouts']}")

Strict:    141/150  (94.0%)
Recovered: 9/150
Invalid:   0/150
Usable structured-output validity (strict+recovered): 100.0%
Total retries: 0   Model errors: 0   Timeouts: 0


## 10. Evidence validity

In [8]:
ev = summary["evidence"]
print("Evidence-bearing cases (Entailment+Contradiction):", ev["evidence_bearing_n"])
print(f"Correct-label-but-invalid-evidence count: {ev['correct_label_bad_evidence_count']}")
print(f"Wrong-label-but-valid-evidence count: {ev['wrong_label_valid_evidence_count']}")
print(f"Paraphrased (non-verbatim) evidence count: {ev['paraphrased_evidence_count']}")

Evidence-bearing cases (Entailment+Contradiction): 100
Correct-label-but-invalid-evidence count: 10
Wrong-label-but-valid-evidence count: 53
Paraphrased (non-verbatim) evidence count: 34


## 11. Evidence Recall / Precision

Computed using ONLY the model's explicitly returned evidence, verbatim-checked and span-mapped
onto the document's real annotated spans -- full-document access is never counted as evidence
success (the historical T041 anti-pattern identified and rejected in E05's Stage A).

In [9]:
print(f"Evidence Recall:    {ev['evidence_recall']:.4f}")
print(f"Evidence Precision:  {ev['evidence_precision']:.4f}")

Evidence Recall:    0.2500
Evidence Precision:  0.2874


## 12. Joint label+evidence success — overall and by class

In [10]:
joint = summary["joint"]
print(f"Overall joint success: {joint['overall']:.4f}")
for label, rate in joint["by_class"].items():
    print(f"  {label:>13}: {rate:.4f}")

Overall joint success: 0.2800
     Entailment: 0.2400
  Contradiction: 0.1600
   NotMentioned: 0.4400


## 9(cont). Input-length analysis — descriptive/correlational only, no causal claim

Buckets: <=p50, p50-p90, >p90 by actual input token count.

In [11]:
lb = summary["length_buckets"]
print(f"p50={lb['p50_tokens']} tokens, p90={lb['p90_tokens']} tokens")
for bucket, stats in lb["stats"].items():
    print(f"\n{bucket} (n={stats['n']}):")
    print(f"  accuracy={stats['accuracy']:.3f}  strict_parse={stats['strict_parse_rate']:.3f}  "
          f"recovered={stats['recovered_parse_rate']:.3f}  invalid={stats['invalid_parse_rate']:.3f}")
    ev_rate = stats['evidence_valid_rate']
    print(f"  evidence_valid_rate={ev_rate:.3f}" if ev_rate is not None else "  evidence_valid_rate=n/a")
    print(f"  mean_latency_ms={stats['mean_latency_ms']:.0f}" if stats['mean_latency_ms'] else "")

p50=2047 tokens, p90=3945 tokens

<=p50 (n=76):
  accuracy=0.395  strict_parse=1.000  recovered=0.000  invalid=0.000
  evidence_valid_rate=0.789
  mean_latency_ms=8363

p50-p90 (n=60):
  accuracy=0.417  strict_parse=0.933  recovered=0.067  invalid=0.000
  evidence_valid_rate=0.750
  mean_latency_ms=14966

>p90 (n=14):
  accuracy=0.357  strict_parse=0.643  recovered=0.357  invalid=0.000
  evidence_valid_rate=0.786
  mean_latency_ms=27215


**Reading this**: any trend across buckets (e.g. strict-parse rate falling or paraphrasing
rising with length) is reported descriptively -- correlation, not a claimed causal long-context
effect, consistent with the reconstruction brief's own instruction.

## 14. Latency

In [12]:
lat = summary["latency_ms"]
print(f"Mean:   {lat['mean']:.0f} ms")
print(f"Median: {lat['median']:.0f} ms")
print(f"P90:    {lat['p90']:.0f} ms")
print(f"Max:    {lat['max']:.0f} ms")
print(f"Total wall time: {wall['total_wall_seconds']/60:.1f} minutes for {summary['n_cases']} cases")

Mean:   12764 ms
Median: 10174 ms
P90:    22214 ms
Max:    84039 ms
Total wall time: 31.9 minutes for 150 cases


## 15. Failure decomposition

Categories per the reconstruction brief (primary+secondary allowed, nothing forced): A
classification/reasoning, B evidence-selection, C structured-output, D long-context-associated
(descriptive only), E exception/carve-out (only if actually observed), F definition/cross-
reference/multi-clause, G NotMentioned overprediction.

In [13]:
ff = summary["failure_family_counts"]
for fam, n in sorted(ff.items(), key=lambda x: -x[1]):
    print(f"{n:>4}  {fam}")

  90  A_classification_reasoning_failure
  40  G_notmentioned_overprediction
  36  E_exception_carveout_candidate
  34  B_evidence_selection_failure


## 16-18. Representative successes and failures

In [14]:
rows = list(csv.DictReader(open(RESULTS / "full_context_failure_analysis.csv")))

print("--- Representative correct predictions ---")
correct = [r for r in rows if r["gold_label"] == r["predicted_label"]][:3]
for r in correct:
    print(f"{r['case_id']}  gold={r['gold_label']}  pred={r['predicted_label']}  "
          f"parse_status={r['parse_status']}  evidence_valid={r['evidence_valid']}")

print("\n--- Representative classification failures (A) ---")
cls_fail = [r for r in rows if "A_classification_reasoning_failure" in r["failure_families"]][:3]
for r in cls_fail:
    print(f"{r['case_id']}  gold={r['gold_label']}  pred={r['predicted_label']}  "
          f"families={r['failure_families']}")

print("\n--- Representative evidence-selection failures (B) ---")
ev_fail = [r for r in rows if "B_evidence_selection_failure" in r["failure_families"]][:3]
for r in ev_fail:
    print(f"{r['case_id']}  gold={r['gold_label']}  pred={r['predicted_label']}  "
          f"evidence_valid={r['evidence_valid']}  evidence_hit={r['evidence_hit']}")

print("\n--- Structured-output failures (C), if any ---")
so_fail = [r for r in rows if "C_structured_output_failure" in r["failure_families"]][:3]
for r in so_fail:
    print(f"{r['case_id']}  gold={r['gold_label']}  parse_status={r['parse_status']}")
if not so_fail:
    print("(none -- all 150 cases were strict or recovered)")

--- Representative correct predictions ---
train::92::nda-17  gold=Contradiction  pred=Contradiction  parse_status=strict  evidence_valid=False
train::92::nda-20  gold=Contradiction  pred=Contradiction  parse_status=strict  evidence_valid=True
train::141::nda-7  gold=Contradiction  pred=Contradiction  parse_status=strict  evidence_valid=False

--- Representative classification failures (A) ---
train::88::nda-1  gold=Contradiction  pred=Entailment  families=A_classification_reasoning_failure;E_exception_carveout_candidate;B_evidence_selection_failure
train::88::nda-2  gold=Contradiction  pred=NotMentioned  families=A_classification_reasoning_failure;G_notmentioned_overprediction;E_exception_carveout_candidate
train::102::nda-1  gold=Contradiction  pred=Entailment  families=A_classification_reasoning_failure;E_exception_carveout_candidate

--- Representative evidence-selection failures (B) ---
train::88::nda-1  gold=Contradiction  pred=Entailment  evidence_valid=False  evidence_hit=False

## 19. E01 Oracle — contextual interpretation only

E01 (Oracle, gold evidence, TRAIN_ORACLE_v1) already established that Qwen's Contradiction
reasoning was weak even under perfect-evidence conditions. E05 tests whether full-document
context changes that behavior on a separate frozen manifest (`TRAIN_ARCH_v1`) — this is NOT a
directly comparable population to E01's, so no exact delta is computed as a causal effect; the
comparison is qualitative only.

In [15]:
print(f"E05 (this experiment) Contradiction Recall: {cls['contradiction_recall']:.1%}")
print("E01 Oracle Contradiction Recall (qwen2.5:7b-instruct, gold evidence, TRAIN_ORACLE_v1):")
print("  see experiments/E01_oracle/summary.md for the exact figure -- different manifest,")
print("  not a matched population, shown for qualitative interpretation only.")

E05 (this experiment) Contradiction Recall: 28.0%
E01 Oracle Contradiction Recall (qwen2.5:7b-instruct, gold evidence, TRAIN_ORACLE_v1):
  see experiments/E01_oracle/summary.md for the exact figure -- different manifest,
  not a matched population, shown for qualitative interpretation only.


## 20. E07 matched-comparison setup

E07 (standard RAG) will use this exact `TRAIN_ARCH_v1` manifest, the same
`classification_prompt_v1`, the same output schema, the same parser
(`evaluation.structured_output.parse_structured_output`), and the same evidence validator —
only the input-construction architecture will differ (`retrieval_v1`'s top-5 reranked context
instead of the full document). Until E07 runs on this same manifest, no "full-context vs. RAG"
claim can be made — E03's earlier retrieved-context result used a different manifest
(`TRAIN_PROMPT_v1`) and is not a valid comparison point for this question.

## 21. Final A1 characterization

`A1_full_context_v1` — frozen, reproducible architecture configuration (not a claim of
production-readiness): full NDA document text input, `classification_prompt_v1`'s system prompt
unchanged + one additive evidence instruction, "Full NDA text:" architecture wrapper, compact
`{"label": ..., "evidence": [...]}` output schema, `qwen2.5:7b-instruct-ctx16k` (num_ctx=16384
baked into the Modelfile), temperature 0.0, 60s timeout, `evaluation.structured_output`
deterministic parser, unmodified `pipeline.evidence_validator`, `TRAIN_ARCH_v1` manifest (150
cases). Full metrics recorded above and in `results/run_E05_A1_train.json`. Findings (weak
Contradiction Recall, any evidence-paraphrasing pattern, any long-context correlation) are
recorded as real results of this configuration -- per instruction, nothing about the prompt,
parser, model, or evidence instruction was changed in response to these results.